# Tracking API (Colab + ngrok)

Backend cho web demo `demo/app.py`. Chạy lần lượt các cell trên Colab (Runtime GPU):

1. Upload checkpoint YOLO26m lên `/content/yolo26m.pt` (hoặc mount Drive rồi sửa đường dẫn ở cell load model).
2. Điền ngrok authtoken vào cell 2.
3. Chạy hết, copy **Public URL** ở cell ngrok dán vào sidebar của Streamlit.

In [ ]:
!pip install fastapi uvicorn pyngrok python-multipart nest_asyncio ultralytics lap

In [ ]:
!ngrok config add-authtoken "<YOUR_NGROK_TOKEN>"

In [ ]:
import os
import threading

import torch
from ultralytics import YOLO

device = "cuda" if torch.cuda.is_available() else "cpu"

MODEL_NAME = "yolo26m"
model = YOLO("/content/yolo26m.pt")  # checkpoint đã train trên VisDrone
print("Loaded:", MODEL_NAME, "| device:", device)
print("Classes:", model.names)

RESULTS_DIR = "/content/results"
os.makedirs(RESULTS_DIR, exist_ok=True)

OUT_MAX_W = 1280  # video kết quả thu nhỏ về rộng tối đa 1280px cho nhẹ (video VisDrone tới 4K)
lock = threading.Lock()  # 1 GPU -> xử lý tuần tự từng request

In [ ]:
import shutil
import subprocess
import time
import uuid
from collections import Counter, defaultdict

import cv2


def to_h264(src, dst):
    """Encode sang H.264 (trình duyệt chỉ phát được codec này), rộng tối đa OUT_MAX_W."""
    subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", src,
                    "-vf", f"scale='min({OUT_MAX_W},iw)':-2", "-c:v", "libx264", "-preset", "veryfast",
                    "-pix_fmt", "yuv420p", "-movflags", "+faststart", "-an", dst], check=True)


def run_tracking(video_path, out_path, tracker, conf, imgsz):
    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS) or 25
    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.release()
    out_w = min(w, OUT_MAX_W)
    out_h = round(h * out_w / w / 2) * 2  # H.264 cần kích thước chẵn
    print(f"[track] {os.path.basename(video_path)}: {w}x{h}, {total} frames -> {out_w}x{out_h}")

    raw_path = out_path.replace(".mp4", "_raw.mp4")
    writer = cv2.VideoWriter(raw_path, cv2.VideoWriter_fourcc(*"mp4v"), fps, (out_w, out_h))

    id_classes = defaultdict(Counter)  # track id -> đếm class qua các frame
    objects_per_frame = []

    # mỗi lần gọi track() với source mới thì tracker được khởi tạo lại -> ID bắt đầu từ 1
    results = model.track(source=video_path, stream=True, tracker=f"{tracker}.yaml",
                          conf=conf, iou=0.5, imgsz=imgsz, device=device, verbose=False)
    for r in results:
        # Ultralytics vẽ sẵn bbox + nhãn "id:N class", màu theo ID
        frame = r.plot(conf=False, line_width=1, color_mode="instance")
        n = 0
        if r.boxes.id is not None:
            ids = r.boxes.id.int().cpu().tolist()
            clss = r.boxes.cls.int().cpu().tolist()
            n = len(ids)
            for tid, c in zip(ids, clss):
                id_classes[tid][model.names[c]] += 1
        objects_per_frame.append(n)

        info = f"Objects: {n} | Unique IDs: {len(id_classes)}"
        cv2.rectangle(frame, (0, 0), (20 + 11 * len(info), 32), (0, 0, 0), -1)
        cv2.putText(frame, info, (10, 22), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2, cv2.LINE_AA)
        if out_w != w:
            frame = cv2.resize(frame, (out_w, out_h), interpolation=cv2.INTER_AREA)
        writer.write(frame)
        if len(objects_per_frame) % 50 == 0:
            print(f"[track] {len(objects_per_frame)}/{total} frames, {len(id_classes)} IDs")
    writer.release()

    # mp4v không phát được trên trình duyệt -> encode lại H.264
    print("[track] encode H.264...")
    to_h264(raw_path, out_path)
    os.remove(raw_path)
    print("[track] done")

    # class của mỗi ID = class xuất hiện nhiều nhất
    per_class = Counter(cnt.most_common(1)[0][0] for cnt in id_classes.values())
    return {
        "unique_ids": len(id_classes),
        "per_class": dict(per_class.most_common()),
        "objects_per_frame": objects_per_frame,
    }

In [ ]:
from fastapi import FastAPI, UploadFile, File, Form
from fastapi.responses import FileResponse

app = FastAPI()

@app.get("/")
def home():
    return {
        "message": "API is running",
        "model": MODEL_NAME,
        "device": torch.cuda.get_device_name(0) if device == "cuda" else "cpu",
    }

@app.post("/track")
def track(
    file: UploadFile = File(...),
    tracker: str = Form("botsort"),
    conf: float = Form(0.1),
    imgsz: int = Form(640),
):
    try:
        if tracker not in ("botsort", "bytetrack"):
            return {"error": f"Tracker không hợp lệ: {tracker}"}

        job_id = uuid.uuid4().hex[:8]
        ext = os.path.splitext(file.filename or "")[1] or ".mp4"
        in_path = os.path.join(RESULTS_DIR, f"{job_id}_input{ext}")
        out_path = os.path.join(RESULTS_DIR, f"{job_id}.mp4")
        with open(in_path, "wb") as f:
            shutil.copyfileobj(file.file, f)

        start = time.time()
        with lock:
            stats = run_tracking(in_path, out_path, tracker, conf, imgsz)
        elapsed = time.time() - start
        # bản H.264 của video gốc để web phát được (video VisDrone gốc là mp4v)
        to_h264(in_path, os.path.join(RESULTS_DIR, f"{job_id}_orig.mp4"))
        os.remove(in_path)

        frames = len(stats["objects_per_frame"])
        return {
            "job_id": job_id,
            "model": MODEL_NAME,
            "tracker": tracker,
            "frames": frames,
            "time_s": round(elapsed, 2),
            "fps": round(frames / elapsed, 2) if elapsed else 0,
            **stats,
        }
    except Exception as e:
        return {"error": str(e)}

@app.get("/video/{job_id}")
def video(job_id: str, original: bool = False):
    # original=true -> video gốc (H.264), mặc định -> video kết quả tracking
    name = f"{job_id}_orig.mp4" if original else f"{job_id}.mp4"
    path = os.path.join(RESULTS_DIR, name)
    if not job_id.isalnum() or not os.path.exists(path):
        return {"error": "Không tìm thấy video"}
    return FileResponse(path, media_type="video/mp4", filename=name)

In [ ]:
from pyngrok import ngrok
import nest_asyncio

nest_asyncio.apply()

public_url = ngrok.connect(8000)
print("Public URL:", public_url)

In [ ]:
import uvicorn
import asyncio

config = uvicorn.Config(app, host="0.0.0.0", port=8000)
server = uvicorn.Server(config)

asyncio.run(server.serve())